# Setup environment

In [18]:
import os
import json

with open(".env.json", "r") as f_in:
    env_json: dict = json.load(f_in)
    for key, value in env_json.items():
        os.environ[key] = value

# Helper Functions

In [70]:
import pandas as pd
import numpy as np

In [43]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name = "cl100k_base") -> int:
    if not string:
        return 0
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

def get_essay_length(essay):
    word_list = essay.split()
    num_words = len(word_list)
    return num_words

def get_embedding_cost(num_tokens):
    return num_tokens/1000*0.0001

def get_total_embeddings_cost(df, column_name_content='content'):
    total_tokens = 0
    for i in df.index:
        text = df[column_name_content][i]
        token_len = num_tokens_from_string(text)
        total_tokens = total_tokens + token_len
    total_cost = get_embedding_cost(total_tokens)
    return total_cost, total_tokens

In [71]:
path_data = "C:/Users/usuario/Documents/GitHub/101MachineLearning/000_data"
region_name = os.environ.get("aws_region", None)
aws_access_key_id = os.environ.get("aws_access_key", None)
aws_secret_access_key = os.environ.get("aws_secret_key", None)
model_embeddings_name = "amazon.titan-embed-text-v1"

postgres_database = os.environ.get("postgres_database", None)
postgres_user = os.environ.get("postgres_user", None)
postgres_password = os.environ.get("postgres_password", None)
postgres_host = os.environ.get("postgres_host", None)
postgres_port = os.environ.get("postgres_port", None)

# Embeddings

In [21]:
import boto3
from langchain.embeddings import BedrockEmbeddings

client_bedrock = boto3.client(
    'bedrock-runtime',
    region_name=region_name,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

In [22]:
embeddings = BedrockEmbeddings(
    model_id=model_embeddings_name,
    client=client_bedrock
)

In [31]:
df_ciiu = pd.read_excel(f"{path_data}/CIIU.xlsx", sheet_name="CIIU 4.0")
df_ciiu.info()

df_ciiu_4 = df_ciiu[["CIIU4_COD", "CIIU4_DESC", "CIIU4_NIVEL"]]
print("CIIU 4")
print(df_ciiu_4.count())
df_ciiu_4.head()

df_ciiu_4_level2 = df_ciiu_4[df_ciiu_4["CIIU4_NIVEL"] == 2].drop_duplicates(["CIIU4_COD"]).reset_index()
print("CIIU 4 - Level 2")
print(df_ciiu_4_level2.count())
df_ciiu_4_level2.head(10)

df_ciiu_4_level6 = df_ciiu_4[df_ciiu_4["CIIU4_NIVEL"] == 6].drop_duplicates(["CIIU4_COD"]).reset_index()
print("CIIU 4 - Level 6")
print(df_ciiu_4_level6.count())
df_ciiu_4_level6.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6919 entries, 0 to 6918
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   CIIU4_COD    6919 non-null   object
 1   CIIU4_DESC   6919 non-null   object
 2   CIIU4_NIVEL  6919 non-null   object
 3   CIIU3_COD    4788 non-null   object
 4   CIIU3_DESC   4788 non-null   object
 5   CIIU3_NIVEL  5852 non-null   object
dtypes: object(6)
memory usage: 324.5+ KB
CIIU 4
CIIU4_COD      6919
CIIU4_DESC     6919
CIIU4_NIVEL    6919
dtype: int64
CIIU 4 - Level 2
CIIU4_COD      91
CIIU4_DESC     91
CIIU4_NIVEL    91
dtype: int64
CIIU 4 - Level 6
CIIU4_COD      1908
CIIU4_DESC     1908
CIIU4_NIVEL    1908
dtype: int64


,CIIU4_COD,CIIU4_DESC,CIIU4_NIVEL
6,A0111.11,Cultivo de trigo.,6
8,A0111.12,Cultivo de maíz.,6
10,A0111.13,Cultivo de quinua.,6
13,A0111.19,"Otros cultivos de cereales n.c.p.: sorgo, ceba...",6
16,A0111.21,Cultivo de fréjol.,6


In [45]:
total_cost, total_tokens = get_total_embeddings_cost(df_ciiu_4_level6, column_name_content="CIIU4_DESC")
print("Precio estimado = $" + str(total_cost) + f" para {total_tokens} tokens")

Precio estimado = $0.0076984 para 76984 tokens


In [54]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1300,
    chunk_overlap  = 20,
    length_function = len,
    add_start_index = True,
)


df_ciiu_4_level6_dummy = df_ciiu_4_level6.iloc[:10, :]
df_ciiu_4_level6_dummy.head(30)


# texts = text_splitter.create_documents([df_ciiu_4_level6["CIIU4_DESC"].values[1]])

,CIIU4_COD,CIIU4_DESC,CIIU4_NIVEL
6,A0111.11,Cultivo de trigo.,6
8,A0111.12,Cultivo de maíz.,6
10,A0111.13,Cultivo de quinua.,6
13,A0111.19,"Otros cultivos de cereales n.c.p.: sorgo, ceba...",6
16,A0111.21,Cultivo de fréjol.,6
19,A0111.22,Cultivo de arveja.,6
21,A0111.23,Cultivo de lenteja.,6
23,A0111.29,"Otros cultivos le legumbres: habas, garbanzos,...",6
26,A0111.31,Cultivo de granos y semillas de soya.,6
29,A0111.32,Cultivo de semillas de maní.,6


In [58]:
documents_embeddings = embeddings.embed_documents(
    list(df_ciiu_4_level6_dummy["CIIU4_DESC"].values)
)

In [65]:
df_ciiu_4_level6_dummy = pd.concat([
    df_ciiu_4_level6_dummy.reset_index(),
    pd.Series(documents_embeddings).to_frame()
], axis=1)

df_ciiu_4_level6_dummy

,index,CIIU4_COD,CIIU4_DESC,CIIU4_NIVEL,0
0,6,A0111.11,Cultivo de trigo.,6,"[-0.6484375, -0.31640625, -0.25, 0.3984375, -1..."
1,8,A0111.12,Cultivo de maíz.,6,"[-0.24414062, -0.07373047, 0.100097656, 0.0532..."
2,10,A0111.13,Cultivo de quinua.,6,"[-0.20996094, -0.20410156, 0.122558594, -0.707..."
3,13,A0111.19,"Otros cultivos de cereales n.c.p.: sorgo, ceba...",6,"[-0.5390625, 0.32421875, 0.012451172, -0.12792..."
4,16,A0111.21,Cultivo de fréjol.,6,"[-0.42773438, 0.69921875, 0.04321289, -0.32421..."
5,19,A0111.22,Cultivo de arveja.,6,"[-0.0008354187, -0.06201172, 0.33203125, -0.14..."
6,21,A0111.23,Cultivo de lenteja.,6,"[-0.4765625, 0.076171875, 0.10449219, 0.163085..."
7,23,A0111.29,"Otros cultivos le legumbres: habas, garbanzos,...",6,"[-0.27539062, 0.38867188, 0.31640625, -0.52734..."
8,26,A0111.31,Cultivo de granos y semillas de soya.,6,"[0.107910156, 0.13378906, 0.096191406, 0.52734..."
9,29,A0111.32,Cultivo de semillas de maní.,6,"[1.0390625, -0.25, -0.06201172, -0.1640625, -0..."


# Postgres - pgvector load

In [72]:
import psycopg2
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values

In [93]:
# Establishing the connection
conn = psycopg2.connect(
    database=postgres_database,
    user=postgres_user,
    password=postgres_password,
    host=postgres_host,
    port=postgres_port
)

In [11]:
cursor = conn.cursor()
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector")
print("Extension enabled........")
conn.commit()

Extension enabled........


In [81]:
cursor = conn.cursor()
 
# Doping EMPLOYEE table if already exists.
cursor.execute("DROP TABLE IF EXISTS public.pg_embeddings_ciiu")
 
# Creating table as per requirement
sql = """CREATE TABLE public.pg_embeddings_ciiu (
    id int GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    content text,
    embedding vector(1536),
    code VARCHAR(255),
    level VARCHAR(2),
    embedding_tool VARCHAR(255),
    date_update TIMESTAMP DEFAULT NOW()
)"""
cursor.execute(sql)

print("Table created successfully........")
conn.commit()

Table created successfully........


In [83]:
cursor = conn.cursor()
cursor.execute("CREATE INDEX ON public.pg_embeddings_ciiu USING ivfflat (embedding vector_cosine_ops)")
print("Table index created successfully........")
conn.commit()

Table index created successfully........


In [74]:
df_ciiu_4_level6_dummy.head()

,index,CIIU4_COD,CIIU4_DESC,CIIU4_NIVEL,0
0,6,A0111.11,Cultivo de trigo.,6,"[-0.6484375, -0.31640625, -0.25, 0.3984375, -1..."
1,8,A0111.12,Cultivo de maíz.,6,"[-0.24414062, -0.07373047, 0.100097656, 0.0532..."
2,10,A0111.13,Cultivo de quinua.,6,"[-0.20996094, -0.20410156, 0.122558594, -0.707..."
3,13,A0111.19,"Otros cultivos de cereales n.c.p.: sorgo, ceba...",6,"[-0.5390625, 0.32421875, 0.012451172, -0.12792..."
4,16,A0111.21,Cultivo de fréjol.,6,"[-0.42773438, 0.69921875, 0.04321289, -0.32421..."


In [82]:
register_vector(conn)
cursor = conn.cursor()
# Prepare the list of tuples to insert
data_list = [(row['CIIU4_DESC'], np.array(row[0]), row['CIIU4_COD'], row['CIIU4_NIVEL'], model_embeddings_name) for index, row in df_ciiu_4_level6_dummy.iterrows()]
# Use execute_values to perform batch insertion
execute_values(cursor, "INSERT INTO public.pg_embeddings_ciiu (content, embedding, code, level, embedding_tool) VALUES %s", data_list)
# Commit after we insert all embeddings
conn.commit()

# Postgres - Vector test

In [103]:
def get_top3_similar_docs(query_embedding, conn, k=3):
    embedding_array = np.array(query_embedding)
    # Register pgvector extension
    register_vector(conn)
    cur = conn.cursor()
    # Get the top 3 most similar documents using the KNN <=> operator
    cur.execute(f"SELECT code, content, embedding <=> %s as distance FROM public.pg_embeddings_ciiu ORDER BY distance LIMIT {k}", (embedding_array,))
    top3_docs = cur.fetchall()
    return top3_docs

In [107]:
user_input = "sembrado de trigo"
related_docs = get_top3_similar_docs(embeddings.embed_query(user_input), conn, k=3)
conn.commit()
related_docs

[('A0111.11', 'Cultivo de trigo.', 0.3875724746710123),
 ('A0111.31', 'Cultivo de granos y semillas de soya.', 0.5379613276151989),
 ('A0111.23', 'Cultivo de lenteja.', 0.5425260389541476)]

# Clean UP

In [108]:
# Closing the connection
conn.close()

# END